# Catalog Creation

Prepare the supplied HUC8 domains and create the replacement catalog using `configs/params-config.json`. Follow the [rerun runbook](../docs/source/lwi_geometry_rerun.rst). The historical catalog remains unchanged.


In [ ]:
from pathlib import Path

from stormhub.logger import initialize_logger
from stormhub.met.catalog_setup import (
    load_catalog_settings,
    prepare_catalog_domains,
    plot_catalog_domains,
    prepare_catalog_inputs,
    create_prepared_catalog,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

SETTINGS_PATH = REPO_ROOT / "configs" / "params-config.json"
settings = load_catalog_settings(SETTINGS_PATH)
CATALOG_ROOT = REPO_ROOT / "catalogs"
CATALOG_DIR = CATALOG_ROOT / settings["catalog_id"]
EXISTING = "error"  # Choose "reuse" to verify/load the same catalog without overwriting.
initialize_logger()
CATALOG_DIR


## Inspect and prepare source geometries

The supplied `.json` files are Esri JSON. Preserve them, verify their pinned bytes, reproject to WGS84, and union all watershed features without simplifying, buffering, or repairing their geometry. StormHub requires one valid Polygon. Preparation fails if that contract cannot be satisfied.


In [ ]:
domains = prepare_catalog_domains(settings, REPO_ROOT)
[domain.summary() for domain in domains.values()]


In [ ]:
plot_catalog_domains(domains, title="LWI Region 3 Watershed and Transposition Region");


## Prepare catalog inputs

The package writes the single-feature GeoJSON inputs, checksums, and a frozen `creation-settings.json` snapshot. Edit `configs/params-config.json` only when preparing a catalog; population reads the selected catalog snapshot directly. `EXISTING="error"` refuses an existing destination. With `"reuse"`, all saved settings/configuration and prepared geometry bytes must match; nothing is rewritten. Changed inputs require a new catalog ID. An incomplete input write is reported for inspection rather than silently repaired.


In [ ]:
config_path = prepare_catalog_inputs(
    settings,
    repository_root=REPO_ROOT,
    catalog_root=CATALOG_ROOT,
    existing=EXISTING,
    config_filename="lwi-r3-config.json",
)
config_path


## Create or reopen the base STAC catalog

For a new catalog this accesses the sample AORC S3 dataset to derive the valid transposition region. With `EXISTING="reuse"`, a completed matching catalog is loaded without writes or an AORC call. If only a base-catalog build was interrupted, reuse can retry it after verifying the prepared inputs and checking that no event products exist. This does not populate storm-event collections.


In [ ]:
catalog = create_prepared_catalog(
    config_path,
    existing=EXISTING,
    description="LWI Region 3 HUC8 geometry revision storm catalog",
)
catalog.spm.catalog_file


In [ ]:
sorted(p.relative_to(CATALOG_DIR) for p in CATALOG_DIR.rglob("*.json"))

## Serve The Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3-geometry-v2 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or use the STAC Browser link shown by the directory listing.